In [1]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from opencv_tools import opencv_tools # 匯入封裝的功能

### 17-2-2 程式範例：讀取影片

In [2]:
# 建立 VideoCapture 物件，指定影片路徑
cap = cv2.VideoCapture('sample/video/cat.mp4')

# 檢查影片是否成功開啟
if not cap.isOpened():
    print('錯誤：無法開啟影片檔案，請檢查路徑或解碼器。')
    exit()

print('成功開啟影片！')

成功開啟影片！


### 17-2-4 程式範例：取得影片資訊

In [3]:
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

print(f'影片總影格數：{total_frames}')
print(f'影片幀率（FPS）：{fps}')
print(f'影片解析度：{width} x {height}')

影片總影格數：423
影片幀率（FPS）：25.0
影片解析度：640 x 338


### 17-2-6 程式範例：逐格讀取影片

In [4]:
while True:
    # 逐格讀取
    ret, frame = cap.read()

    # 如果讀取失敗（例如影片播完了），就跳出迴圈
    if not ret:
        print("影片播放結束或讀取失敗。")
        break

    # 顯示影格
    cv2.imshow('Video Playback', frame)

    # 控制播放速度（waitKey 等待 25 毫秒再播下一格），按 q 鍵可手動離開
    if cv2.waitKey(25) & 0xFF == ord('q'):
        break

# 釋放資源與關閉視窗
cap.release()
cv2.destroyAllWindows()

影片播放結束或讀取失敗。


### 17-3-2 程式範例：即時輪廓偵測與繪製

In [5]:
def process_frame(frame):
    # 轉為灰階
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # 二值化處理（門檻值設為 127）
    _, thresh = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)

    # 偵測輪廓
    contours, _ = cv2.findContours(
        thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # 在原圖的副本上繪製紅色輪廓
    contour_img = frame.copy()
    cv2.drawContours(contour_img, contours, -1, (0, 0, 255), 2)

    return contour_img

In [6]:
# 主程式讀取影片
cap = cv2.VideoCapture('sample/video/cat.mp4')

if not cap.isOpened():
    print('無法開啟影片檔案，請檢查路徑。')
else:
    while True:
        # 逐格讀取
        ret, frame = cap.read()
        if not ret: # 影片結束
            break 
        # 呼叫自訂的處理函式
        result_frame = process_frame(frame)
        # 顯示結果
        cv2.imshow('Contours Real-time', result_frame)
        # 控制播放速度與退出機制
        if cv2.waitKey(25) & 0xFF == ord('q'):
            break

    # 釋放資源
    cap.release()
    cv2.destroyAllWindows()

### 17-4-3 程式範例：將處理後的影片儲存為新檔案

In [7]:
# 主程式讀取影片
cap = cv2.VideoCapture('sample/video/cat.mp4')

if not cap.isOpened():
    print('無法開啟影片')
else:
    # 取得原始影片資訊
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # 建立 VideoWriter 物件
    fourcc = cv2.VideoWriter_fourcc(*'XVID')
    out = cv2.VideoWriter('output_contours.avi', fourcc, fps, (width, height))

    while True:
        ret, frame = cap.read() # 逐格讀取
        if not ret: # 影片結束
            break
            
        # 處理影像並寫入檔案
        result_frame = process_frame(frame)
        out.write(result_frame)

        # 逐格顯示處理結果 (順便表示處理進度)
        cv2.imshow('Processing and Saving...', result_frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
       
    # 釋放資源 (這對儲存檔案非常關鍵)
    cap.release()
    out.release()
    cv2.destroyAllWindows()
    print("影片儲存完成！")

影片儲存完成！


### 17-E-1 程式範例：製作影片負片（Negative）效果並儲存

In [8]:
def process_frame(frame):
    return cv2.bitwise_not(frame)

In [9]:
# 主程式讀取影片
cap = cv2.VideoCapture('sample/video/cat.mp4')

if not cap.isOpened():
    print('錯誤：無法開啟影片檔案。')
else:
    # 1. 取得原始影片資訊並轉為整數
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # 2. 建立 VideoWriter 物件 (注意儲存彩色影片，isColor 為 True)
    fourcc = cv2.VideoWriter_fourcc(*'XVID')
    out = cv2.VideoWriter('output_negative.avi', fourcc, fps, (width, height), isColor=True)

    print("[INFO] 開始處理影片...")

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # 3. 呼叫負片處理函式並寫入檔案
        negative_result = process_frame(frame)
        out.write(negative_result)

        # 逐格顯示處理結果 (順便表示處理進度)
        cv2.imshow('Negative Video Effect', negative_result)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    # 4. 釋放資源
    cap.release()
    out.release()
    cv2.destroyAllWindows()
    print("[INFO] 負片影片儲存完成：output_negative.avi")

[INFO] 開始處理影片...
[INFO] 負片影片儲存完成：output_negative.avi
